In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from matplotlib import pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
import string
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression,SGDClassifier
from sklearn.svm import LinearSVC, OneClassSVM
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from scipy.sparse import hstack, csr_matrix, vstack
from sklearn.decomposition import TruncatedSVD
from sklearn.mixture import GaussianMixture
from sklearn.feature_selection import SelectKBest, chi2
from scipy.stats import uniform


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


RANDOM_STATE = 42
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

##Milestone 1


# Data Loading

In [ ]:
sample = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/Sample.csv")
sample

In [ ]:
train = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/train.csv")
train_df = train.copy()
train_df

In [ ]:
train_df.isna().sum()

In [ ]:
# imputer = SimpleImputer(strategy="most_frequent")

In [ ]:
# train_df[["gender"]] = imputer.fit_transform(train_df[["gender"]])

In [ ]:
test = pd.read_csv("/kaggle/input/comment-category-prediction-challenge/test.csv")
test_df = test.copy()
test_df

In [ ]:
train_df['label'].value_counts()

In [ ]:
sns.countplot(x='label', data=train_df)
plt.title("Label Distribution")
plt.show()

### Labels are imbalanced

In [ ]:
X_raw = train_df.drop(columns=['label'])
y = train_df['label']

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
train_eda = X_train_raw.copy()
train_eda['label'] = y_train

In [ ]:
X_val_raw.info()

In [ ]:
y.shape

# Identifing Data Types 

In [ ]:
sample.info()

In [ ]:
X_train_raw.info()

In [ ]:
test_df.info()

# Descriptive Statistics of Numerical Columns

In [ ]:
X_train_raw.describe().T

In [ ]:
test_df.describe().T

# Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
sns.histplot(X_train_raw['upvote'], bins=50)
plt.title("Upvotes Distribution")

plt.subplot(1,2,2)
sns.histplot(X_train_raw['downvote'], bins=50)
plt.title("Downvotes Distribution")

plt.show()# we can see both the distribution are right skewed

In [ ]:
X_train_raw['comment'] = X_train_raw['comment'].fillna("")
X_train_raw['char_len'] = X_train_raw['comment'].str.len()

X_val_raw['comment'] = X_val_raw['comment'].fillna("")
X_val_raw['char_len'] = X_val_raw['comment'].str.len()

test_df['comment'] = test_df['comment'].fillna("")
test_df['char_len'] = test_df['comment'].str.len()
avg_len = X_train_raw.groupby(y)['char_len'].mean()

plt.figure(figsize=(6,4))
sns.barplot(x=avg_len.index, y=avg_len.values)

plt.title("Average Comment Length per Label")
plt.xlabel("Label")
plt.ylabel("Average Length")

plt.show()

In [ ]:
from collections import Counter

for l in sorted(train_eda['label'].unique()):
    text = " ".join(train_eda[train_eda['label']==l]['comment'].fillna(""))
    words = text.lower().split()
    print(f"\nLabel {l} top words:")
    print(Counter(words).most_common(10))

In [ ]:
emote_avg = train_eda.groupby('label')[['emoticon_1','emoticon_2','emoticon_3']].mean()

emote_avg.plot(kind='bar', figsize=(8,4))

plt.title("Emoticon Usage per Label")
plt.xlabel("Label")
plt.ylabel("Average Count")

plt.show()

In [ ]:
avg_if1 = train_eda.groupby('label')['if_1'].mean()
avg_if2 = train_eda.groupby('label')['if_2'].mean()

plt.figure(figsize=(8,4))

plt.bar(avg_if1.index - 0.2, avg_if1.values, width=0.4, label='if_1')
plt.bar(avg_if2.index + 0.2, avg_if2.values, width=0.4, label='if_2')

plt.title("if_1 and if_2 vs Label")
plt.xlabel("Label")
plt.ylabel("Average Value")
plt.legend()

plt.show()

In [ ]:
X_train_raw['char_len'] = X_train_raw['comment'].fillna("").str.len()
X_val_raw['char_len'] = X_val_raw['comment'].fillna("").str.len()
test_df['char_len'] = test_df['comment'].fillna("").str.len()

plt.figure(figsize=(6,4))

plt.scatter(
    X_train_raw['char_len'],
    X_train_raw['upvote'],
    alpha=0.3
)

plt.title("Comment Length vs Upvotes")
plt.xlabel("Comment Length")
plt.ylabel("Upvotes")

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.scatter(
    X_train_raw['upvote'],
    X_train_raw['downvote'],
    c=y_train,
    cmap='viridis',
    alpha=0.3
)

plt.colorbar(label="Label")
plt.title("Upvotes vs Downvotes")
plt.xlabel("Upvotes")
plt.ylabel("Downvotes")
plt.show()

In [ ]:
X_train_raw['vote_ratio'] = X_train_raw['upvote'] / (X_train_raw['downvote'] + 1)
test_df['vote_ratio'] = test_df['upvote'] / (test_df['downvote'] + 1)
X_val_raw['vote_ratio'] = X_val_raw['upvote'] / (X_val_raw['downvote'] + 1)

avg_ratio = X_train_raw.groupby(y_train)['vote_ratio'].mean()

plt.figure(figsize=(6,4))
plt.bar(avg_ratio.index, avg_ratio.values)

plt.title("Average Vote Ratio per Label")
plt.xlabel("Label")
plt.ylabel("Vote Ratio")

plt.show()

In [ ]:
X_train_raw['log_upvote'] = np.log1p(X_train_raw['upvote'])
test_df['log_upvote'] = np.log1p(test_df['upvote'])
X_val_raw['log_upvote'] = np.log1p(X_val_raw['upvote'])


avg_up = X_train_raw.groupby(y_train)['log_upvote'].mean()

plt.figure(figsize=(6,4))
sns.barplot(x=avg_up.index, y=avg_up.values)

plt.title("Log Upvotes vs Label")
plt.xlabel("Label")
plt.ylabel("Avg Log Upvote")

plt.show()

In [ ]:
test_df.shape

In [ ]:
X_train_raw.shape

In [ ]:
X_val_raw.shape

# Handling Null Values

In [ ]:
X_train_raw.isnull().sum()

In [ ]:
X_val_raw.isnull().sum()

In [ ]:
test_df.isna().sum()

In [ ]:
print("NaN count:", X_train_raw["race"].isna().sum())

print("Empty string count:", (X_train_raw["race"] == "").sum())

print("Whitespace count:", X_train_raw["race"].str.strip().eq("").sum())

In [ ]:
print("NaN count:", X_train_raw["religion"].isna().sum())

print("Empty string count:", (X_train_raw["religion"] == "").sum())

print("Whitespace count:", X_train_raw["religion"].str.strip().eq("").sum())

In [ ]:
print("NaN count:", X_train_raw["gender"].isna().sum())

print("Empty string count:", (X_train_raw["gender"] == "").sum())

print("Whitespace count:", X_train_raw["gender"].str.strip().eq("").sum())

In [ ]:
print("NaN count:", X_train_raw["created_date"].isna().sum())

print("Empty string count:", (X_train_raw["created_date"] == "").sum())

print("Whitespace count:", X_train_raw["created_date"].str.strip().eq("").sum())

In [ ]:
print("NaN count:", X_train_raw["comment"].isna().sum())

print("Empty string count:", (X_train_raw["comment"] == "").sum())

print("Whitespace count:", X_train_raw["comment"].str.strip().eq("").sum())

In [ ]:
y_train.value_counts()

In [ ]:
y.isna().sum()

In [ ]:
num_cols = X_train_raw.select_dtypes(include=['int64','float64','bool']).columns.to_list()

In [ ]:
num_cols

In [ ]:
cat_cols = X_train_raw.select_dtypes(include=['object']).columns.to_list()

In [ ]:
cat_cols

In [ ]:
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

In [ ]:
X_train_raw[cat_cols] = cat_imputer.fit_transform(X_train_raw[cat_cols])
X_val_raw[cat_cols] = cat_imputer.transform(X_val_raw[cat_cols])
test_df[cat_cols] = cat_imputer.transform(test_df[cat_cols])

X_train_raw[num_cols] = num_imputer.fit_transform(X_train_raw[num_cols])
X_val_raw[num_cols] = num_imputer.transform(X_val_raw[num_cols])
test_df[num_cols] = num_imputer.transform(test_df[num_cols])

In [ ]:
X_train_raw.isna().sum()

In [ ]:
X_val_raw.isna().sum()

In [ ]:
test_df.isna().sum()

# Handling Duplicates

In [ ]:
test_df.duplicated().sum()

In [ ]:

X_train_raw.duplicated().sum()

In [ ]:
X_val_raw.duplicated().sum()

* There are no Duplicate Data Points in the Dataset.

In [ ]:
# Milestone 2 Q2 — month frequency

train_df["created_date"] = pd.to_datetime(train_df["created_date"])

month_counts = train_df["created_date"].dt.month_name().str.lower().value_counts()

month_counts.head()


In [ ]:
month_counts.idxmax()

In [ ]:
X_train_raw.info()

### Median Number Of Upvote per comment

In [ ]:
train_df["upvote"].median()

### Numerical feature shows the largest maximum value

In [ ]:
train_df.select_dtypes(include="number").max().sort_values(ascending=False)

### Minimum value of if_2

In [ ]:
train_df['if_2'].min()

In [ ]:
# Milestone 2 Q3 — total_emoticons feature

X_train_raw["total_emoticons"] = (
    X_train_raw["emoticon_1"] +
    X_train_raw["emoticon_2"] +
    X_train_raw["emoticon_3"]
)

X_val_raw["total_emoticons"] = (
    X_val_raw["emoticon_1"] +
    X_val_raw["emoticon_2"] +
    X_val_raw["emoticon_3"]
)

test_df["total_emoticons"] = (
    test_df["emoticon_1"] +
    test_df["emoticon_2"] +
    test_df["emoticon_3"]
)
X_train_raw["total_emoticons"].max()

In [ ]:
# Milestone 2 Q4

subset = train_df.copy()

subset['comment'] = subset['comment'].fillna("")

subset['char_len'] = subset['comment'].str.len()

subset.loc[subset['label']==3, 'char_len'].median()

minmax formula 

(x - min) / (max - min)

In [ ]:
# Milestone 2 Q5

min_up = train_df['upvote'].min()
max_up = train_df['upvote'].max()

scaled_10 = (10 - min_up) / (max_up - min_up)

scaled_10

In [ ]:
# Milestone 2 Q6

df_q6 = train_df.copy()

df_q6['comment'] = df_q6['comment'].fillna("")

df_q6['word_count'] = df_q6['comment'].str.split().str.len()

avg_wc = df_q6.loc[df_q6['label']==1, 'word_count'].mean()

round(avg_wc, 2)

In [ ]:
X_train_raw['comment'] = X_train_raw['comment'].str.strip().fillna("")

X_val_raw['comment'] = X_val_raw['comment'].str.strip().fillna("")

test_df['comment'] = test_df['comment'].str.strip().fillna("")

In [ ]:
# Milestone 2 Q7
df_q7 = train_df.copy()

df_q7['comment'] = df_q7['comment'].fillna("")

df_q7['comment'].str.contains("trump", case=False).sum()

In [ ]:
# Milestone 2 Q8

text = train_df.loc[0, 'comment']

if pd.isna(text):
    text = ""

text = text.lower()

text = text.translate(str.maketrans('', '', string.punctuation))

words = text.split()

stopwords = [
'a','an','the','and','or','but','if','because','as','of','at','by','for','with','about',
'to','from','up','on','in','out','over','under','is','are','was','were','be','been','being',
'have','has','had','do','does','did','it','its','they','them','their','she','her','he','him',
'his','this','that','which','who','whom','i','me','my','we','our','you','your'
]

filtered = [w for w in words if w not in stopwords]

len(filtered)

In [ ]:
# Milestone 2 Q9

df_m = train_df.copy()

df_m['comment'] = df_m['comment'].fillna("").str.lower()

tokens = df_m['comment'].str.split()

unique_tokens = set()

for t in tokens:
    unique_tokens.update(t)

len(unique_tokens)

# Handling Outliers

In [ ]:
for col in num_cols:
    Q1 = X_train_raw[col].quantile(0.25)
    Q3 = X_train_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((X_train_raw[col] < lower) | (X_train_raw[col] > upper)).sum()
    print(f"{col}:        {outliers}")

In [ ]:
train_eda = X_train_raw.copy()
train_eda['label'] = y_train

In [ ]:
n_cols = 2
n_rows = (len(num_cols) + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(
        data=train_eda,
        x='label',
        y=col,
        ax=axes[i],
        showfliers=True
    )
    axes[i].set_title(f"{col} by Label (Outliers)")
    axes[i].set_xlabel("label")
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=train_eda[num_cols])
plt.xticks(rotation=45)
plt.title("Boxplots of Numeric Features (Outliers Visible)")
plt.tight_layout()
plt.show()


In [ ]:
for col in num_cols:
    Q1 = X_train_raw[col].quantile(0.25)
    Q3 = X_train_raw[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before = ((X_train_raw[col] < lower) | (X_train_raw[col] > upper)).sum()

    X_train_raw[col] = X_train_raw[col].clip(lower=lower, upper=upper)
    X_val_raw[col]   = X_val_raw[col].clip(lower=lower, upper=upper)
    test_df[col]     = test_df[col].clip(lower=lower, upper=upper)

    print(f"Feature: {col:15} | Range: [{lower:.2f}, {upper:.2f}] applied.")

    after = ((X_train_raw[col] < lower) | (X_train_raw[col] > upper)).sum()

    # print(f"{col}: capped {before} values, remaining outliers = {after}")

In [ ]:
train_eda = X_train_raw.copy()
train_eda['label'] = y_train

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=train_eda[num_cols])
plt.xticks(rotation=45)
plt.title("Boxplots of Numeric Features (Outliers Visible)")
plt.tight_layout()
plt.show()

## Correlation – Visualizations

In [ ]:
num_df = train_df.select_dtypes(include=['int64','float64','bool', 'int32'])

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    train_eda.corr(numeric_only=True),
    cmap='coolwarm',
    annot=False,
    linewidths=0.5
)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

In [ ]:
remove_cols = ['emoticon_1','emoticon_2','emoticon_3','disability']
corr = train_eda[num_df.columns].drop(columns=remove_cols).corr()
corr

In [ ]:
plt.figure(figsize=[12,8])
sns.heatmap(data=corr, annot=True, cmap='coolwarm', fmt='.2g')
plt.show()

In [ ]:
X_train_raw.describe().T

# Feature Scaling and Encoding

In [ ]:
X_train_raw['created_date'] = pd.to_datetime(X_train_raw['created_date'])
X_val_raw['created_date'] = pd.to_datetime(X_val_raw['created_date'])
test_df['created_date'] = pd.to_datetime(test_df['created_date'])

for df in [X_train_raw, X_val_raw, test_df]:
    df['year'] = df['created_date'].dt.year
    df['month'] = df['created_date'].dt.month
    df['day'] = df['created_date'].dt.day
    df['hour'] = df['created_date'].dt.hour

X_train_raw.drop(columns=['created_date'], inplace=True)
X_val_raw.drop(columns=['created_date'], inplace=True)
test_df.drop(columns=['created_date'], inplace=True)

In [ ]:
X_train_raw.info()

In [ ]:
test_df.info()

In [ ]:
cat_cols = X_train_raw.select_dtypes(include=['object']).columns
cat_cols


In [ ]:
X_train_raw['race'].value_counts()

In [ ]:
X_train_raw['religion'].value_counts()

In [ ]:
X_train_raw['gender'].value_counts()

In [ ]:
cat_cols = ['race', 'religion', 'gender']

X_train_raw = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True)
X_val_raw = pd.get_dummies(X_val_raw, columns=cat_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=True)

In [ ]:
num_cols = X_train_raw.select_dtypes(include=['int64', 'float64', 'int32', 'bool']).columns.tolist()

num_cols

In [ ]:
X_train_raw.info()

In [ ]:
num_cols

In [ ]:
X_train_raw.shape

In [ ]:
X_val_raw.shape

In [ ]:
test_df.shape

In [ ]:
scaler = RobustScaler()

X_num_train = scaler.fit_transform(X_train_raw[num_cols])
X_num_val = scaler.transform(X_val_raw[num_cols])
X_num_test = scaler.transform(test_df[num_cols])

X_num_train_sparse = csr_matrix(X_num_train)
X_num_val_sparse = csr_matrix(X_num_val)
X_num_test_sparse = csr_matrix(X_num_test)

In [ ]:
X_num_test_sparse.shape

In [ ]:
X_num_train_sparse.shape

In [ ]:
X_num_val_sparse.shape

In [ ]:
# num_cols = [
#     "emoticon_1", "emoticon_2", "emoticon_3",
#     "upvote", "downvote", "if_1", "if_2", "disability"
# ]
num = X_train_raw.select_dtypes(include=['int64','float64', 'int32','bool']).columns.to_list()

X_vif = X_train_raw[num].fillna(0)
X_vif = RobustScaler().fit_transform(X_vif)
X_vif = pd.DataFrame(X_vif, columns=num)

vif_df = pd.DataFrame()
vif_df["Feature"] = X_vif.columns
vif_df["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

vif_df.sort_values("VIF", ascending=False)

In [ ]:
X_train_raw["disability"].value_counts()

In [ ]:
X_train_raw['emoticon_1'].value_counts()

In [ ]:
X_train_raw['emoticon_2'].value_counts()

In [ ]:
X_train_raw['emoticon_3'].value_counts()

In [ ]:
X_train_raw['total_emoticons'].value_counts()

In [ ]:
X_train_raw['total_emoticons'].describe()

In [ ]:
X_train_raw['total_emoticons'] = np.log1p(X_train_raw['total_emoticons'])
X_val_raw['total_emoticons'] = np.log1p(X_val_raw['total_emoticons'])
test_df['total_emoticons'] = np.log1p(test_df['total_emoticons'])

In [ ]:
X_train_raw['total_emoticons'].describe()

In [ ]:
X_train_raw['total_emoticons'].value_counts()

In [ ]:
drop_cols = ['disability', 'emoticon_1', 'emoticon_2', 'emoticon_3']

X_train_raw.drop(columns=drop_cols, inplace=True)
X_val_raw.drop(columns=drop_cols, inplace=True)
test_df.drop(columns=drop_cols, inplace=True)

In [ ]:
num

In [ ]:
X_train_raw.info()

In [ ]:
num = X_train_raw.select_dtypes(include=['int64','float64', 'int32','bool']).columns.to_list()

X_vif = X_train_raw[num].fillna(0)
X_vif = RobustScaler().fit_transform(X_vif)
X_vif = pd.DataFrame(X_vif, columns=num)

vif_df = pd.DataFrame()
vif_df["Feature"] = X_vif.columns
vif_df["VIF"] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

vif_df.sort_values("VIF", ascending=False)

In [ ]:
# Milestone 2 Q10

vec = TfidfVectorizer(
    stop_words="english",
    min_df=5,
    ngram_range=(1,2)
)

X_tmp = vec.fit_transform(train_df['comment'].fillna(""))

len(vec.get_feature_names_out())

In [ ]:
# Precision = tp/tp+fp
#recall = tp/tp+fn
#Accuracy = (TP+TN)/(TP+TN+FP+FN)
#F1 = (2*Recall*precision)/Precision+Recall

In [ ]:
#minmax = A+(X-min/max-min)(b-a)

In [ ]:
# "i am avi"    , i = 1, 
# "i am a boy"
# "i am a toy"

In [ ]:
tfidf_word = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    max_features=200000, 
    stop_words='english',
    lowercase=True,
    sublinear_tf=True,
    norm="l2"
)
X_word_train = tfidf_word.fit_transform(X_train_raw['comment'])
X_word_val = tfidf_word.transform(X_val_raw['comment'])
X_word_test = tfidf_word.transform(test_df['comment'])

tfidf_char = TfidfVectorizer(
    analyzer='char', 
    ngram_range=(4, 6), 
    min_df=3,
    max_features=200000 
)
X_char_train = tfidf_char.fit_transform(X_train_raw['comment'])
X_char_val = tfidf_char.fit_transform(X_val_raw['comment'])
X_char_test = tfidf_char.transform(test_df['comment'])

X_text_train = hstack([X_word_train, X_char_train])
X_text_val = hstack([X_word_val, X_char_val])
X_text_test = hstack([X_word_test, X_char_test])

X_train = hstack([X_num_train, X_text_train])
X_val = hstack([X_num_val, X_text_val])
X_test = hstack([X_num_test, X_text_test])

In [ ]:
X_train_full = hstack([X_text_train, X_num_train_sparse])
X_val_full = hstack([X_text_val, X_num_val_sparse])
X_test_full = hstack([X_text_test, X_num_test_sparse])

X_train_full.shape, X_test_full.shape, X_val_full.shape

In [ ]:
k = 220000
selector = SelectKBest(score_func=chi2, k=k)

X_text_train_sel = selector.fit_transform(X_text_train, y_train)
X_text_val_sel   = selector.transform(X_text_val)
X_text_test_sel  = selector.transform(X_text_test)

print("Selected Training Shape:", X_text_train_sel.shape)

In [ ]:
X_text_val_sel.shape

In [ ]:
X_text_test_sel.shape

In [ ]:
X_train_sel = hstack([X_text_train_sel, X_num_train_sparse])
X_val_sel   = hstack([X_text_val_sel,   X_num_val_sparse])
X_test_sel  = hstack([X_text_test_sel,  X_num_test_sparse])

print(f"Final Train: {X_train_sel.shape}")
print(f"Final Val:   {X_val_sel.shape}")
print(f"Final Test:  {X_test_sel.shape}")

# Model Building

In [ ]:
results = {}
trained_models = {}

In [ ]:
logreg = LogisticRegression(max_iter=4000,
                            penalty="l2",
                            C=2.5,
                            solver="saga",
                            class_weight='balanced',
                            n_jobs=-1,
                            tol=1e-4,
                            random_state=RANDOM_STATE)

logreg.fit(X_train_sel, y_train)
f1 = f1_score(y_val, logreg.predict(X_val_sel),average="macro")

results["Logistic Regression"] = f1
trained_models["Logistic Regression"] = logreg

print("Logistic Regression Macro-F1:", f1)

In [ ]:
svm = LinearSVC(max_iter=2000, C=0.15, class_weight="balanced", random_state=RANDOM_STATE)
svm.fit(X_train_sel, y_train)

f1 = f1_score(y_val, svm.predict(X_val_sel), average="macro")

results["LinearSVC"] = f1
trained_models["LinearSVC"] = svm

print("LinearSVC Macro-F1:", f1)

In [ ]:
sgd_log = SGDClassifier(loss="log_loss",
                          penalty="l2",
                          alpha=1e-5,
                          max_iter=2000,
                          random_state=RANDOM_STATE,
                          n_jobs=-1)
sgd_log.fit(X_train_sel, y_train)
sgd_log_pred = sgd_log.predict(X_val_sel)
f1 = f1_score(y_val, sgd_log_pred, average="macro")

results["SGD (Log)"] = f1
trained_models["SGD (Log)"] = sgd_log

print("SGD (Log) Macro-F1:", f1)

In [ ]:
voting = VotingClassifier(
    estimators=[
        ("lr", logreg),
        ("svm", svm),
        ("sgd_log", sgd_log)
    ],
    voting="hard",
    n_jobs=-1
)

voting.fit(X_train_sel, y_train)
f1 = f1_score(y_val, voting.predict(X_val_sel), average="macro")

results["Voting Classifier"] = f1
trained_models["Voting Classifier"] = voting

print("Voting Classifier Macro-F1:", f1)

# Hyperparameter Tuning


In [ ]:
C_values = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 2.5, 2.9, 3.0, 3.1, 3.5, 4.0]

best_f1 = -1
best_lr = None
best_lr_C = None

for C in C_values:
    lr = LogisticRegression(
        C=C,
        max_iter=4000,
        penalty="l2",
        solver="saga",
        class_weight="balanced",
        n_jobs=-1,
        tol=1e-4,
        random_state=RANDOM_STATE
    )
    lr.fit(X_train_sel, y_train)
    preds = lr.predict(X_val_sel)
    f1 = f1_score(y_val, preds, average="macro")
    print(f"C={C} → Macro-F1={f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_lr = lr
        best_lr_C = C

print("Best C:", best_lr_C)
print("Best Macro-F1:", best_f1)
results["LogisticRegression (tuned)"] = best_f1
trained_models["LogisticRegression (tuned)"] = best_lr

In [ ]:
Cs = [0.01, 0.05, 0.08, 0.085, 0.09, 0.095, 0.1, 0.13, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]

best_svm_f1 = -1
best_svm_model = None
best_svm_C = None

print("Tuning Linear SVM (Macro-F1)...")

for C in Cs:
    svm = LinearSVC(
        C=C,
        class_weight="balanced",
        random_state=42
    )
    
    svm.fit(X_train_sel, y_train)
    y_val_pred = svm.predict(X_val_sel)

    f1 = f1_score(y_val, y_val_pred, average="macro")
    print(f"C={C} → Macro-F1 = {f1:.4f}")

    if f1 > best_svm_f1:
        best_svm_f1 = f1
        best_svm_model = svm
        best_svm_C = C

print("\nBest C:", best_svm_C)
print("Best Linear SVM Macro-F1:", best_svm_f1)

results["LinearSVC (tuned)"] = best_svm_f1
trained_models["LinearSVC (tuned)"] = best_svm_model

In [ ]:
svm_cal = CalibratedClassifierCV(
    estimator=best_svm_model,
    method="sigmoid",
    cv=3
)

svm_cal.fit(X_train_sel, y_train)

In [ ]:
alphas = [1e-5, 1e-4, 5e-5, 1e-7, 1e-6]

best_sgd_f1 = -1
best_sgd_model = None
best_sgd_alpha = None

print("Tuning SGDClassifier...")


for alpha in alphas:
    sgd = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=alpha,
        max_iter=1000,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    sgd.fit(X_train_sel, y_train)
    y_val_pred = sgd.predict(X_val_sel)

    f1 = f1_score(y_val, y_val_pred, average="macro")
    print(f"alpha={alpha} --> Macro-F1 = {f1:.4f}")

    if f1 > best_sgd_f1:
        best_sgd_f1 = f1
        best_sgd_model = sgd
        best_sgd_alpha = alpha

print("\nBest SGD alpha:", best_sgd_alpha)
print("Best SGD Validation Macro-F1:", best_sgd_f1)

results["SGDClassifier (tuned)"] = best_sgd_f1
trained_models["SGDClassifier (tuned)"] = best_sgd_model


In [ ]:
best_alpha_sgd = best_sgd_alpha
best_C_svm = best_svm_C     
best_C_lr  = best_lr_C


base_voting = VotingClassifier(
    estimators=[
        ("lr", trained_models["LogisticRegression (tuned)"]),
        ("svm_cal", CalibratedClassifierCV(
            LinearSVC(C=best_C_svm),
            method="sigmoid",
            cv=3
        )),
        ("sgd", trained_models["SGDClassifier (tuned)"])
        ],
    voting="soft",
    n_jobs=-1
)

In [ ]:
weight_configs = [
    (2, 1, 1),
    (3, 1, 1),
    (3, 2, 1)
]

best_voting_f1 = -1
best_voting_model = None
best_voting_weights = None

print("Tuning Soft VotingClassifier (Macro-F1)...")

for weights in weight_configs:
    voting_clf = VotingClassifier(
        estimators=[
            ("lr", best_lr),
            ("svm", svm_cal),
            ("sgd", best_sgd_model)
        ],
        voting="soft",
        weights=weights,
        n_jobs=-1
    )

    voting_clf.fit(X_train_sel, y_train)
    y_val_pred = voting_clf.predict(X_val_sel)

    f1 = f1_score(y_val, y_val_pred, average="macro")
    print(f"Weights={weights} → Macro-F1={f1:.4f}")

    if f1 > best_voting_f1:
        best_voting_f1 = f1
        best_voting_model = voting_clf
        best_voting_weights = weights

print("\nBest Voting Weights:", best_voting_weights)
print("Best Voting Macro-F1:", best_voting_f1)

results["VotingClassifier (tuned)"] = best_voting_f1
trained_models["VotingClassifier (tuned)"] = best_voting_model

# Comparison Table for Models

In [ ]:
results_df = pd.DataFrame({
    "Model": list(results.keys()),
    "Validation Macro-F1": list(results.values())
})

results_df = results_df.sort_values(by="Validation Macro-F1", ascending=False)
results_df.reset_index(drop=True, inplace=True)

results_df

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(results_df["Model"], results_df["Validation Macro-F1"], color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.title("Model Performance Comparison")
plt.ylabel("Validation Accuracy")
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

# Confusion Matrix Plot

In [ ]:
best_row = results_df.loc[results_df["Validation Macro-F1"].idxmax()]
best_model_name = best_row["Model"]
best_val_f1 = best_row["Validation Macro-F1"]

print("Best model by Validation Macro-F1:", best_model_name)
print("Stored Validation Macro-F1:", best_val_f1)

final_model = trained_models[best_model_name]

X_val_final = X_val_sel

y_val_pred_final = final_model.predict(X_val_final)
final_f1 = f1_score(y_val, y_val_pred_final, average="macro")

print(f"\nFinal Model Validation Macro-F1: {final_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred_final))

cm = confusion_matrix(y_val, y_val_pred_final)
cm

In [ ]:
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Confusion Matrix – {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# Fit on Full Data and Predict Test

In [ ]:
best_row = results_df.iloc[0]

best_model_name = best_row["Model"]
best_macro_f1 = best_row["Validation Macro-F1"]


In [ ]:
best_model = trained_models[best_model_name]

In [ ]:
print("Best model:", best_model_name)
print("Best validation Macro-F1:", best_macro_f1)
print(best_model)

In [ ]:
# 1. Combine Text Data (Vertical Stack)
X_text_full = vstack([X_text_train, X_text_val])
X_text_full_sel = selector.transform(X_text_full)

# 2. Combine Numerical Data (Vertical Stack)
# Ensure X_num_val_sparse was created in the scaling section
X_num_full = vstack([X_num_train_sparse, X_num_val_sparse])

# 3. Combine Horizontal
X_full_final = hstack([X_text_full_sel, X_num_full])

# 4. Fit on ALL 198,000 labels
# Use the full labels from original train_df
y_full = train_df['label'].values 
best_model.fit(X_full_final, y_full)

print(f"Model refitted on full dataset. Shape: {X_full_final.shape}")

# 5. Predict on Test 
X_text_test = hstack([X_word_test, X_char_test])
X_text_test_sel = selector.transform(X_text_test)
X_test_final = hstack([X_text_test_sel, X_num_test_sparse])

test_pred = best_model.predict(X_test_final)
print(f"Generated {len(test_pred)} predictions for submission.")

# Submission File

In [ ]:
submission = pd.DataFrame({
    "ID": np.arange(1, len(test_pred) + 1),
    "label": test_pred
})

submission.to_csv("submission.csv", index=False)
submission

In [ ]:
submission['label'].value_counts()

In [ ]:
sample